In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from warnings import filterwarnings
filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import hf_hub_download

In [ ]:
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
roberta_base = AutoModel.from_pretrained("roberta-base")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device: ", device)

In [ ]:
roberta_base.pooler = None
roberta_base.gradient_checkpointing_enable()

In [ ]:
roberta_base.config.hidden_size

In [ ]:
test_path = "/content/drive/MyDrive/SemEval-Test.csv"

df_test = pd.read_csv(test_path)

In [ ]:
class SemEval_Dataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer):
        self.tokenizer = tokenizer
        self.data = data
        self.max_len = 128
        self.target_cols = [str(i) for i in range(11)]

    def __len__(self):
        return(len(self.data))

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        text = str(item.text)
        encoding = self.tokenizer.encode_plus(text,
                                            add_special_tokens=True,
                                            truncation=True,
                                            return_tensors='pt',
                                            max_length=self.max_len,
                                            padding='max_length',
                                            return_attention_mask=True)

        target = torch.tensor(item[self.target_cols].values.astype('float32'))

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "atten_mask": encoding["attention_mask"].squeeze(0),
            "hard_target": target
        }

In [ ]:
test_dataloader = DataLoader(SemEval_Dataset(df_test, roberta_tokenizer), batch_size=64, num_workers=2)

In [ ]:
class Encoder(nn.Module):

    def __init__(self, base_encoder):
        super().__init__()
        self.encoder = base_encoder

    def forward(self, inputs):

        outputs = self.encoder(**inputs, output_hidden_states=True)
        last_hidden_state = outputs.hidden_states[-1]                                                            # [B, T, H]

        atten_mask = inputs['attention_mask']                                                                    # [B, T]

        atten_mask = atten_mask.unsqueeze(-1).float()
        text_emb = (last_hidden_state * atten_mask).sum(dim=1) / atten_mask.sum(dim=1).clamp(min=1e-9)           # [B, H]
        text_emb = F.normalize(text_emb, p=2, dim=1)

        return text_emb

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=11):
        super().__init__()
        self.input_dim = input_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes)
        )

    def forward(self, h):
        return self.mlp(h)

In [ ]:
# Main Model class
class EmoAxis(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, inputs: dict):
        # Encoder
        outputs = self.encoder(inputs)

        # Classifier
        logits = self.classifier(outputs)

        return outputs, logits

In [ ]:
def evaluate(model, dataloader, device, threshold=0.5):

    model.eval()

    preds_all = []
    truths_all = []

    with torch.no_grad():
        for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['atten_mask'].to(device)
            hard_target = batch['hard_target'].to(device)

            _,logits = model(inputs={"input_ids": input_ids, "attention_mask": attention_mask})

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            preds_all.append(preds.cpu())
            truths_all.append(hard_target.cpu().int())

    preds_all = torch.cat(preds_all, dim=0).numpy()
    truths_all = torch.cat(truths_all, dim=0).numpy()

    # Compute metrics
    micro_precision = precision_score(truths_all, preds_all, average='micro', zero_division=0)
    macro_precision = precision_score(truths_all, preds_all, average='macro', zero_division=0)

    micro_recall = recall_score(truths_all, preds_all, average='micro', zero_division=0)
    macro_recall = recall_score(truths_all, preds_all, average='macro', zero_division=0)

    micro_f1 = f1_score(truths_all, preds_all, average='micro', zero_division=0)
    macro_f1 = f1_score(truths_all, preds_all, average='macro', zero_division=0)

    print(f"\n\nMicro Precision: {micro_precision} \nMacro Precision: {macro_precision}\n")
    print(f"Micro Recall: {micro_recall} \nMacro Recall: {macro_recall}\n")
    print(f"Micro F1: {micro_f1} \nMacro F1: {macro_f1}")

    emotion_labels = [
    "anger", "anticipation", "disgust", "fear", "joy", "love",
    "optimism", "pessimism", "sadness", "surprise", "trust"]

    print("\n===== CLASSIFICATION REPORT =====\n")

    print(classification_report(
        truths_all,
        preds_all,
        target_names=emotion_labels,
        zero_division=0
    ))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = hf_hub_download(
    repo_id="Hidden-States/roberta-base-semeval-pt-only",
    filename="EmoAxis-SemEval.pt"
)

In [ ]:
checkpoint = torch.load(ckpt_path, map_location="cpu")
state_dict = checkpoint["model_state_dict"]

In [ ]:
encoder = Encoder(base_encoder=roberta_base)
classifier = Classifier()
trained_model = EmoAxis(encoder=encoder, classifier=classifier)

In [ ]:
trained_model.load_state_dict(state_dict, strict=False)
trained_model.to(device)
print("Checkpoint loaded successfully!")

In [ ]:
trained_model.eval()

In [ ]:
evaluate(trained_model, test_dataloader, device)